# B2-019 — Session 1: Query, Key, Value, and Scaled Dot Product

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F5-probability`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).


## 1. Transpose and shape ledger

For $Q\in\mathbb R^{n_q\times d_k}$ and $K\in\mathbb R^{n_k\times d_k}$, $K^\top$ has shape $d_k\times n_k$. Therefore the multiplication $QK^\top$ lies in $\mathbb R^{n_q\times n_k}$, and entry $(i,j)$ is the dot product $q_i\cdot k_j$. The row index names the destination query; the column index names the candidate source key. In a batch, only the last two axes multiply, so `(B,n_q,d_k) @ (B,d_k,n_k)` gives `(B,n_q,n_k)`.

**Worked example 1.** With $Q=[[1,2]]$ and keys $k_0=(3,0)$, $k_1=(0,4)$, $QK^\top=(3,8)$. This writes every scalar multiply explicitly before any normalization.

**Checkpoint 1A.** Trace $(B,n_q,d_k)@(B,d_k,n_k)$.

**Checkpoint 1B.** Which two axes does `swapaxes(-1,-2)` exchange?

In [ ]:
import numpy as np
SEED = 20260808
ATOL = 1e-10
RTOL = 1e-10
Q = np.array([[1., 0.], [0., 1.]])
K = np.array([[1., 0.], [1., 1.], [0., 1.]])
assert (Q @ K.T).shape == (2, 3)

## 2. Query, key, and value roles

A query represents what a destination position seeks; a key represents what each source position offers for matching; a value carries the content mixed after matching. Q and K must share $d_k$ because their feature axis contracts. K and V must share $n_k$ because every key has one associated value row. The value width $d_v$ may differ, so attention separates *where to read* from *what is returned*.

**Checkpoint 2A.** Which object determines the output row count?

**Checkpoint 2B.** Which object determines the output feature width?

In [ ]:
def stable_row_softmax(scores):
    shifted = scores - scores.max(axis=-1, keepdims=True)
    numer = np.exp(shifted)
    return numer / numer.sum(axis=-1, keepdims=True)
weights = stable_row_softmax(np.array([[np.log(2.0), 0.0]]))
assert np.allclose(weights, [[2/3, 1/3]], atol=ATOL, rtol=RTOL)

## 3. Scale and stable row softmax

Raw scores are $S=QK^\top/\sqrt{d_k}$. If independent coordinates have mean zero and variance one, a dot product of $d_k$ terms has variance $d_k$; division by $\sqrt{d_k}$ returns the variance to order one. For stable softmax, subtract the maximum in each row before exponentiating: $A_{ij}=\exp(S_{ij}-m_i)/\sum_t\exp(S_{it}-m_i)$. This changes neither the ratios nor the answer, but prevents large positive scores from overflowing.

**Worked example 2.** If one scaled row is $(\log 2,0)$, the shifted exponentials are $(1,1/2)$ and the weights are $(2/3,1/3)$.

**Checkpoint 3A.** Why divide by $\sqrt{d_k}$?

**Checkpoint 3B.** Why normalize each query row independently?

In [ ]:
V = np.array([[2., 0.], [0., 4.]])
O = np.array([[0.25, 0.75]]) @ V
assert np.allclose(O, [[0.5, 3.0]], atol=ATOL, rtol=RTOL)

## 4. Weighted values

The output is $O=AV$. Since $A\in\mathbb R^{n_q\times n_k}$ and $V\in\mathbb R^{n_k\times d_v}$, $O\in\mathbb R^{n_q\times d_v}$. Each output row is a convex combination of source value rows: weights are nonnegative and have row sum one. Attention may mix source content, but it cannot create a feature width not supplied by V.

**Worked example 3.** Weights $(1/4,3/4)$ and values $(2,0)$ and $(0,4)$ yield $(1/2,3)$.

**Checkpoint 4A.** State the output shape.

**Checkpoint 4B.** What happens when a weight row is one-hot?

## 5. End-to-end invariant audit

An implementation should check the entire chain, not only the final shape: finite numeric inputs; compatible Q/K and K/V dimensions; scores of shape `(B,n_q,n_k)`; stable softmax on the key axis; finite nonnegative weights; row sums equal to one; and output shape `(B,n_q,d_v)`. Exact small probes catch a transposed score matrix even when square inputs hide the mistake.

**Checkpoint 5A.** Why is `n_q != n_k` a useful shape probe?

**Checkpoint 5B.** Which two invariants distinguish weights from arbitrary scores?

## 6. Common pitfalls

Broken: multiply $QK$ without transposing keys. Fix: make the contracting axes both $d_k$. Broken: one softmax over the full matrix. Fix: normalize the key axis per query. Broken: infer correctness from a square output alone. Fix: certify an exact nonsquare probe and the row-stochastic invariants.

**Exam connections.** Round 2 questions often compress the entire computation into a small exact matrix and require both shape and arithmetic.

**Going deeper.** Session 2 adds masks without changing this pipeline.

Checkpoint answers: 1A $(B,n_q,n_k)$; 1B the last two axes; 2A Q; 2B V; 3A variance control; 3B each query forms its own distribution; 4A $(n_q,d_v)$; 4B it copies one value row; 5A it exposes a Q/K swap; 5B nonnegativity and row sums of one.